In [ ]:
!pip install tenseal

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# -*- coding: utf-8 -*-
"""
Script Otimizado para Teste da PoC (Fase 3): Criptografia Homomórfica com TenSEAL/CKKS
Modelo: Autoencoder Tabular (PyTorch) + Aprendizado Federado (FedAvg) Paralelizado
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import concurrent.futures
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.nn.utils import parameters_to_vector, vector_to_parameters

import tenseal as ts

# ==========================================
# NOVO: FUNÇÃO PARA FIXAR O DETERMINISMO
# ==========================================
def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        # Forçar operações determinísticas na GPU (pode reduzir performance)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

# Aplicar a semente antes de qualquer coisa
set_seed(42)

# Configurações de estilo e dispositivo
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 5)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo PyTorch: {device}")


print("\n--- 1. Carregamento e Preparação dos Dados ---\n")
try:
    # Atualizado o caminho do CSV para o Google Drive
    file_path = '/content/drive/MyDrive/Mestrado Cesar/Base de Dados/Kaggle/paysim.csv'
    df = pd.read_csv(file_path)

    # Sub-sample para FHE (Foco em manter os dados de fraude e uma amostra de normais)
    if len(df) > 100000:
        print("Realizando amostragem (100k linhas) para viabilidade do FHE...")
        df_fraudes = df[df['isFraud'] == 1]
        df_normais = df[df['isFraud'] == 0].sample(n=(100000 - len(df_fraudes)), random_state=42)
        df = pd.concat([df_normais, df_fraudes]).sample(frac=1, random_state=42).reset_index(drop=True)

except FileNotFoundError:
    raise FileNotFoundError(f"ERRO: O arquivo '{file_path}' não foi encontrado. Verifique se o Google Drive está montado e se o caminho está correto.")

df = df.drop(columns=['nameOrig', 'nameDest', 'isFlaggedFraud']).dropna(subset=['isFraud'])

print("Aplicando Engenharia de Features...")
df['amount_log'] = np.log1p(df['amount'])
df['errorBalanceOrig'] = df['newbalanceOrig'] + df['amount'] - df['oldbalanceOrg']

# One-Hot Encoding e separação
df_ml = pd.get_dummies(df, columns=['type'], drop_first=True)
X = df_ml.drop(columns=['isFraud']).astype(np.float32)
y = df_ml['isFraud'].astype(np.int32)

# Correção de Data Leakage: Split antes de escalar
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X.values, y.values, test_size=0.2, random_state=42, stratify=y.values
)

# Fit apenas no X_train, transform no X_train e X_test
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

input_dim = X_train.shape[1]
print(f"Dimensão de entrada (Features): {input_dim}")


class TabularAutoencoder(nn.Module):
    def __init__(self, input_dim):
        super(TabularAutoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 8),
            nn.ReLU()
        )
        # OMITIDO a Sigmoid() no decoder pois o StandardScaler gera valores fora de [0,1]
        self.decoder = nn.Sequential(
            nn.Linear(8, input_dim)
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))


print("\n--- 2. Setup de Criptografia Homomórfica (TenSEAL/CKKS) ---\n")
print("Gerando chaves criptográficas FHE (Isso pode levar alguns segundos)...")
context = ts.context(
    ts.SCHEME_TYPE.CKKS,
    poly_modulus_degree=8192,
    coeff_mod_bit_sizes=[60, 40, 40, 60]
)
context.global_scale = 2**40
context.generate_galois_keys()
print("-> Contexto CKKS gerado com sucesso!")


print("\n--- 3. Aprendizado Federado Paralelizado (Treinamento Local) ---\n")
NUM_CLIENTES = 3

# Filtra apenas dados normais para treinar o Autoencoder
X_train_normal = X_train[y_train == 0]
tamanho_chunk = len(X_train_normal) // NUM_CLIENTES
clientes_X = [X_train_normal[i * tamanho_chunk : (i + 1) * tamanho_chunk] for i in range(NUM_CLIENTES)]

modelo_global = TabularAutoencoder(input_dim).to(device)
criterion = nn.MSELoss()

def treinar_cliente_e_criptografar(cliente_id, dados_X, state_dict_global, epochs=3, batch_size=64):
    """Função que roda em paralelo: Treina, achata os pesos nativamente e os encripta."""
    inicio = time.time()

    # 1. Setup local
    modelo_local = TabularAutoencoder(input_dim).to(device)
    modelo_local.load_state_dict(state_dict_global)
    modelo_local.train()
    optimizer = optim.Adam(modelo_local.parameters(), lr=0.01)

    # Otimização: Pin_memory acelera transferãncia para GPU
    dataset = TensorDataset(torch.tensor(dados_X, dtype=torch.float32), torch.tensor(dados_X, dtype=torch.float32))

    # NOVO: Garantir que o DataLoader use a mesma seed para o shuffle
    generator = torch.Generator(device='cpu') # Generator para dataloader sempre no CPU
    generator.manual_seed(42 + cliente_id) # Seed ónica mas determinística por cliente

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        pin_memory=(device.type == 'cuda'),
        generator=generator # Aplica o generator determinístico
    )

    # 2. Treinamento
    for _ in range(epochs):
        for data, target in loader:
            data, target = data.to(device, non_blocking=True), target.to(device, non_blocking=True)
            optimizer.zero_grad()
            output = modelo_local(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()

    # 3. Extração Otimizada de Pesos (Vetorização C++ nativa do PyTorch)
    pesos_planos = parameters_to_vector(modelo_local.parameters()).detach().cpu().numpy().tolist()

    # 4. Criptografia Homomórfica
    pesos_cifrados = ts.ckks_vector(context, pesos_planos)

    tempo_total = time.time() - inicio
    print(f"[Cliente {cliente_id+1}] Treino e Criptografia concluídos em {tempo_total:.2f}s.")
    return pesos_cifrados


# Usando ThreadPoolExecutor para simular os clientes operando simultaneamente
pesos_criptografados_clientes = []
state_dict_atual = {k: v.cpu() for k, v in modelo_global.state_dict().items()}

with concurrent.futures.ThreadPoolExecutor(max_workers=NUM_CLIENTES) as executor:
    futures = [
        executor.submit(treinar_cliente_e_criptografar, i, clientes_X[i], state_dict_atual)
        for i in range(NUM_CLIENTES)
    ]
    for future in concurrent.futures.as_completed(futures):
        pesos_criptografados_clientes.append(future.result())

print("\n--- 4. Agregação Federada CEGA (Servidor Central) ---\n")
print("Servidor calculando operações sobre dados cifrados (FHE)...")
inicio_agregracao = time.time()

# Soma homomórfica
soma_cifrada = pesos_criptografados_clientes[0]
for i in range(1, NUM_CLIENTES):
    soma_cifrada += pesos_criptografados_clientes[i]

# Média homomórfica
media_cifrada = soma_cifrada * (1.0 / NUM_CLIENTES)

tempo_agreg = time.time() - inicio_agregracao
print(f"-> Agregação matemática FHE finalizada em {tempo_agreg:.2f}s.")


print("\n--- 5. Descriptografia e Inferência ---\n")
# Cliente decifra e restaura a topologia
pesos_planos_agregados = media_cifrada.decrypt()

# Restauração otimizada dos pesos para o modelo global
tensor_pesos = torch.tensor(pesos_planos_agregados, dtype=torch.float32).to(device)
vector_to_parameters(tensor_pesos, modelo_global.parameters())
print("-> Modelo Global atualizado com os pesos agregados!")

# AVALIAÇÃO
modelo_global.eval()
inicio_inferencia = time.time()

with torch.no_grad():
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
    reconstrucoes = modelo_global(X_test_tensor)
    erros_reconstrucao = torch.mean((X_test_tensor - reconstrucoes) ** 2, dim=1).cpu().numpy()

fim_inferencia = time.time()
tempo_inferencia = fim_inferencia - inicio_inferencia

# Limiar de anomalia (Fraude) fixado no Percentil 95
threshold = np.percentile(erros_reconstrucao, 95)
y_pred = (erros_reconstrucao > threshold).astype(int)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_test, erros_reconstrucao)

print("\n>> MÉTRICAS DO MODELO COM CRIPTOGRAFIA FHE (CKKS):")
print(f"Acurácia Geral      : {acc * 100:.2f}%")
print(f"Precisão (Precision): {prec * 100:.2f}%")
print(f"Revocação (Recall)  : {rec * 100:.2f}%")
print(f"F1-Score            : {f1 * 100:.2f}%")
print(f"ROC-AUC             : {roc_auc:.4f}")
print(f"Throughput de Infe. : {len(X_test)/tempo_inferencia:.2f} TPS")

# Plotagem de Resultados
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribuição de Erros
sns.histplot(erros_reconstrucao[y_test == 0], color='#2ecc71', bins=50, alpha=0.6, label='Normal', ax=axes[0])
sns.histplot(erros_reconstrucao[y_test == 1], color='#e74c3c', bins=50, alpha=0.6, label='Fraude', ax=axes[0])
axes[0].axvline(threshold, color='black', linestyle='--', linewidth=2, label='Limiar (Threshold)')
axes[0].set_yscale('log')
axes[0].set_title('Erro de Reconstrução FHE (Escala Log)', fontsize=12)
axes[0].set_xlabel('Erro Quadrático Médio (MSE)')
axes[0].set_ylabel('Frequência (Log)')
axes[0].legend()

# Matriz de Confusão
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=['Normal', 'Fraude'], yticklabels=['Normal', 'Fraude'],
            cbar=False, annot_kws={"size": 14})
axes[1].set_title('Matriz de Confusão (Agregação Segura)', fontsize=12)
axes[1].set_xlabel('Predito')
axes[1].set_ylabel('Real')

plt.tight_layout()
plt.show()

print("\nFim do processamento (Fase 3 Otimizada)!")

In [ ]:
!pip install tenseal

In [ ]:
# -*- coding: utf-8 -*-
"""
Script Otimizado para Teste da PoC (Fase 3): Criptografia Homomórfica com TenSEAL/CKKS
Dataset: Credit Card Fraud (Kaggle)
Modelo: Autoencoder Tabular (PyTorch) + Aprendizado Federado (FedAvg) Paralelizado
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import concurrent.futures
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.nn.utils import parameters_to_vector, vector_to_parameters

try:
    import tenseal as ts
except ImportError:
    print("ERRO: TenSEAL não está instalado no seu ambiente.")
    print("Por favor, execute: pip install tenseal")
    exit()

# ==========================================
# FUNÇÃO PARA FIXAR O DETERMINISMO
# ==========================================
def set_seed(seed=42):
    import random
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        # Forçar operações determinísticas na GPU (pode reduzir performance)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

# Aplicar a semente antes de qualquer coisa
set_seed(42)

# Configurações de estilo e dispositivo
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 5)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo PyTorch: {device}")

print("\n--- 1. Carregamento e Preparação dos Dados ---\n")
file_path = '/content/drive/MyDrive/Mestrado Cesar/Base de Dados/creditcard/creditcard.csv'

try:
    df = pd.read_csv(file_path)
    print("Arquivo 'creditcard.csv' carregado com sucesso!")
except FileNotFoundError:
    print(f"Aviso: O arquivo '{file_path}' não foi encontrado.")
    print("Criando dados sintéticos no formato Credit Card para simulação FHE.")
    # Fallback sintético
    df = pd.DataFrame(np.random.randn(10000, 28), columns=[f'V{i}' for i in range(1, 29)])
    df['Time'] = np.random.uniform(0, 172792, 10000)
    df['Amount'] = np.abs(np.random.randn(10000)) * 100
    df['Class'] = np.random.choice([0, 1], 10000, p=[0.995, 0.005])

# Sub-sample para FHE (Foco em manter os dados de fraude e uma amostra de normais)
# O CKKS e a vetorização de pesos da rede neural podem consumir muita RAM.
# É uma boa prática reduzir a base normal para testes rápidos de conceito.
if len(df) > 100000:
    print("Realizando amostragem (100k linhas) para viabilidade do FHE...")
    df_fraudes = df[df['Class'] == 1]
    df_normais = df[df['Class'] == 0].sample(n=(100000 - len(df_fraudes)), random_state=42)
    df = pd.concat([df_normais, df_fraudes]).sample(frac=1, random_state=42).reset_index(drop=True)

# Remover Time (Ruído para autoencoder tabular)
if 'Time' in df.columns:
    df = df.drop(columns=['Time'])

df = df.dropna(subset=['Class'])

print("Aplicando Engenharia de Features...")
# Proteção para o log1p
if 'Amount' in df.columns:
    df['Amount'] = df['Amount'].apply(lambda x: max(x, 0))
    df['amount_log'] = np.log1p(df['Amount'])
    df = df.drop(columns=['Amount'])

# Separação
X = df.drop(columns=['Class']).values.astype(np.float32)
y = df['Class'].values.astype(np.int32)

# Correção de Data Leakage: Split Duplo (Treino, Validação, Teste)
X_temp, X_test_raw, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp # 0.25 de 0.8 = 0.2
)

# Fit apenas no X_train, transform no X_train, X_val e X_test
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_val = scaler.transform(X_val_raw)
X_test = scaler.transform(X_test_raw)

input_dim = X_train.shape[1]
print(f"Dimensão de entrada (Features): {input_dim}")

class TabularAutoencoder(nn.Module):
    def __init__(self, input_dim):
        super(TabularAutoencoder, self).__init__()
        # Para base PCA de cartão de crédito (~29 features), uma compressão simples para FHE
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 14),
            nn.ReLU(),
            nn.Linear(14, 8),
            nn.ReLU()
        )
        # OMITIDO a Sigmoid() no decoder pois o StandardScaler gera valores fora de [0,1]
        self.decoder = nn.Sequential(
            nn.Linear(8, 14),
            nn.ReLU(),
            nn.Linear(14, input_dim)
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

print("\n--- 2. Setup de Criptografia Homomórfica (TenSEAL/CKKS) ---\n")
print("Gerando chaves criptográficas FHE (Isso pode levar alguns segundos)...")
# O grau do polinômio (8192) dita a segurança e a capacidade de packing (quantos pesos cabem em um cipher).
context = ts.context(
    ts.SCHEME_TYPE.CKKS,
    poly_modulus_degree=8192,
    coeff_mod_bit_sizes=[60, 40, 40, 60]
)
context.global_scale = 2**40
context.generate_galois_keys()
print("-> Contexto CKKS gerado com sucesso!")

print("\n--- 3. Aprendizado Federado Paralelizado (Treinamento Local) ---\n")
NUM_CLIENTES = 3

# Filtra apenas dados normais para treinar o Autoencoder
X_train_normal = X_train[y_train == 0]
tamanho_chunk = len(X_train_normal) // NUM_CLIENTES
clientes_X = [X_train_normal[i * tamanho_chunk : (i + 1) * tamanho_chunk] for i in range(NUM_CLIENTES)]

modelo_global = TabularAutoencoder(input_dim).to(device)
criterion = nn.MSELoss()

def treinar_cliente_e_criptografar(cliente_id, dados_X, state_dict_global, epochs=3, batch_size=64):
    """Função que roda em paralelo: Treina, achata os pesos nativamente e os encripta."""
    inicio = time.time()

    # 1. Setup local
    modelo_local = TabularAutoencoder(input_dim).to(device)
    modelo_local.load_state_dict(state_dict_global)
    modelo_local.train()
    optimizer = optim.Adam(modelo_local.parameters(), lr=0.01)

    # Otimização: Pin_memory acelera transferência para GPU
    dataset = TensorDataset(torch.tensor(dados_X, dtype=torch.float32), torch.tensor(dados_X, dtype=torch.float32))

    # NOVO: Garantir que o DataLoader use a mesma seed para o shuffle
    generator = torch.Generator(device='cpu') # Generator para dataloader sempre no CPU
    generator.manual_seed(42 + cliente_id) # Seed única mas determinística por cliente

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        pin_memory=(device.type == 'cuda'),
        generator=generator # Aplica o generator determinístico
    )

    # 2. Treinamento
    for _ in range(epochs):
        for data, target in loader:
            data, target = data.to(device, non_blocking=True), target.to(device, non_blocking=True)
            optimizer.zero_grad()
            output = modelo_local(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()

    # 3. Extração Otimizada de Pesos (Vetorização C++ nativa do PyTorch)
    pesos_planos = parameters_to_vector(modelo_local.parameters()).detach().cpu().numpy().tolist()

    # 4. Criptografia Homomórfica
    pesos_cifrados = ts.ckks_vector(context, pesos_planos)

    tempo_total = time.time() - inicio
    print(f"[Cliente {cliente_id+1}] Treino e Criptografia concluídos em {tempo_total:.2f}s.")
    return pesos_cifrados

# Usando ThreadPoolExecutor para simular os clientes operando simultaneamente
pesos_criptografados_clientes = []
state_dict_atual = {k: v.cpu() for k, v in modelo_global.state_dict().items()}

with concurrent.futures.ThreadPoolExecutor(max_workers=NUM_CLIENTES) as executor:
    futures = [
        executor.submit(treinar_cliente_e_criptografar, i, clientes_X[i], state_dict_atual)
        for i in range(NUM_CLIENTES)
    ]
    for future in concurrent.futures.as_completed(futures):
        pesos_criptografados_clientes.append(future.result())

print("\n--- 4. Agregação Federada CEGA (Servidor Central) ---\n")
print("Servidor calculando operações sobre dados cifrados (FHE)...")
inicio_agregracao = time.time()

# Soma homomórfica
soma_cifrada = pesos_criptografados_clientes[0]
for i in range(1, NUM_CLIENTES):
    soma_cifrada += pesos_criptografados_clientes[i]

# Média homomórfica
media_cifrada = soma_cifrada * (1.0 / NUM_CLIENTES)

tempo_agreg = time.time() - inicio_agregracao
print(f"-> Agregação matemática FHE finalizada em {tempo_agreg:.2f}s.")

print("\n--- 5. Descriptografia e Inferência ---\n")
# Cliente decifra e restaura a topologia
pesos_planos_agregados = media_cifrada.decrypt()

# Restauração otimizada dos pesos para o modelo global
tensor_pesos = torch.tensor(pesos_planos_agregados, dtype=torch.float32).to(device)
vector_to_parameters(tensor_pesos, modelo_global.parameters())
print("-> Modelo Global atualizado com os pesos agregados!")

print("\n--- 5.1 Otimização de Limiar (Base de Validação) ---\n")
modelo_global.eval()

with torch.no_grad():
    X_val_tensor = torch.tensor(X_val, dtype=torch.float32).to(device)
    reconstrucoes_val = modelo_global(X_val_tensor)
    erros_reconstrucao_val = torch.mean((X_val_tensor - reconstrucoes_val) ** 2, dim=1).cpu().numpy()

# Limiar de anomalia (Fraude) fixado no Percentil 98 USANDO APENAS A VALIDAÇÃO
threshold = np.percentile(erros_reconstrucao_val, 98)
print(f"Limiar de Anomalia Definido (P98 na Validação): {threshold:.4f}")

print("\n--- 5.2 AVALIAÇÃO FINAL (Base de Teste) ---\n")
inicio_inferencia = time.time()

with torch.no_grad():
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
    reconstrucoes = modelo_global(X_test_tensor)
    erros_reconstrucao = torch.mean((X_test_tensor - reconstrucoes) ** 2, dim=1).cpu().numpy()

fim_inferencia = time.time()
tempo_inferencia = fim_inferencia - inicio_inferencia

# Aplicando o limiar descoberto na validação sobre os dados de teste
y_pred = (erros_reconstrucao > threshold).astype(int)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_test, erros_reconstrucao)

print("\n>> MÉTRICAS DO MODELO COM CRIPTOGRAFIA FHE (CKKS):")
print(f"Acurácia Geral      : {acc * 100:.2f}%")
print(f"Precisão (Precision): {prec * 100:.2f}%")
print(f"Revocação (Recall)  : {rec * 100:.2f}%")
print(f"F1-Score            : {f1 * 100:.2f}%")
print(f"ROC-AUC             : {roc_auc:.4f}")
print(f"Throughput de Infe. : {len(X_test)/tempo_inferencia:.2f} TPS")

# Plotagem de Resultados
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribuição de Erros
sns.histplot(erros_reconstrucao[y_test == 0], color='#2ecc71', bins=50, alpha=0.6, label='Normal', ax=axes[0])
sns.histplot(erros_reconstrucao[y_test == 1], color='#e74c3c', bins=50, alpha=0.6, label='Fraude', ax=axes[0])
axes[0].axvline(threshold, color='black', linestyle='--', linewidth=2, label='Limiar (Threshold)')
axes[0].set_yscale('log')
axes[0].set_title('Erro de Reconstrução FHE (Escala Log)', fontsize=12)
axes[0].set_xlabel('Erro Quadrático Médio (MSE)')
axes[0].set_ylabel('Frequência (Log)')
axes[0].legend()

# Matriz de Confusão
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=['Normal', 'Fraude'], yticklabels=['Normal', 'Fraude'],
            cbar=False, annot_kws={"size": 14})
axes[1].set_title('Matriz de Confusão (Agregação Segura)', fontsize=12)
axes[1].set_xlabel('Predito')
axes[1].set_ylabel('Real')

plt.tight_layout()
plt.show()

print("\nFim do processamento (Fase 3 Otimizada para Credit Card Fraud)!")